In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from torch import Tensor
from torch import nn
from torch.nn.parameter import Parameter
import torch.nn.functional as F 
from uci_tests.iris.iris import Iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import torch.optim as optim

In [7]:
sample = 20
rules = 5
inp = 10
mid = 3

t1 = torch.rand((sample, inp, rules))
t2 = torch.rand((inp, mid, rules))
t1 = t1.permute(2, 0, 1)
t2 = t2.permute(2, 0, 1)

t3 = t1 @ t2
print(t3.shape)

torch.Size([5, 20, 3])


In [19]:
class FQ(nn.Module):
    def __init__(self, in_features: int, rules: int, out_features: int, device=None, dtype=None, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        factory_kwargs = {'device': device, 'dtype': dtype}
        self.mean = Parameter(torch.rand((1, in_features, rules), **factory_kwargs))
        self.std = Parameter(torch.rand((1, in_features, rules), **factory_kwargs))
        self.tt = Parameter(torch.randn((1, in_features, rules), **factory_kwargs) * 0.1)
        self.linear = nn.Linear(in_features=rules, out_features=out_features)
        
    def forward(self, X: Tensor):
        y = X.unsqueeze(2)
        tt = torch.tanh(self.tt)
        y = y - self.mean
        y = y / self.std
        y = torch.exp(-torch.pow(y, 2))
        y = (y * tt) + 0.5 * (1 - tt)
        y = torch.prod(y, dim=1)
        f_sum = torch.sum(y, dim=1) 
        f_sum = torch.where(f_sum == 0, f_sum, 1).unsqueeze(dim=1)
        y = y / f_sum
        y = self.linear(y)
        y = torch.softmax(y, dim=1)
        return y
        


In [20]:
# X, y = Iris().get_data()

In [28]:
from uci_tests.sonar.sonar import Sonar
from uci_tests.balance.balance import Balance
from uci_tests.seismic.seismic import Seismic
from uci_tests.haberman.haberman import Haberman
X, y = Haberman().get_data()

In [30]:
from sklearn.model_selection import KFold, train_test_split
import torch.optim as optim

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

# Initialize KFold with 5 folds
kf = KFold(n_splits=20, shuffle=True, random_state=50)

# Define batch size
batch_size = 32

# Initialize lists to store accuracies for each fold
accuracies = []

# Iterate over each fold
for train_index, test_index in kf.split(X):
    # Split data into train and test sets for this fold
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Convert data to PyTorch tensors and move to GPU
    X_train = torch.Tensor(X_train).to(device)
    X_test = torch.Tensor(X_test).to(device)
    y_train = torch.Tensor(y_train).argmax(dim=1).to(device)
    y_test = torch.Tensor(y_test).argmax(dim=1).to(device)

    # Initialize and move model to GPU
    model = FQ(in_features=X.shape[1], rules=4, out_features=y.shape[1]).to(device)
    
    # Initialize Adam optimizer
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # Mini-batch training loop
    num_epochs = 500
    
    model.train()
    for epoch in range(num_epochs):
        for i in range(0, len(X_train), batch_size):
            optimizer.zero_grad()
            batch_X, batch_y = X_train[i:i+batch_size], y_train[i:i+batch_size]
            outputs = model(batch_X)
            loss = F.cross_entropy(outputs, batch_y)
            loss.backward()
            optimizer.step()
    
    # Evaluate the model on the test set and calculate accuracy
    model.eval()
    with torch.no_grad():
        prediction = model(X_test).argmax(dim=1)
        accuracy = accuracy_score(y_test.cpu().numpy(), prediction.cpu().numpy())
        accuracies.append(accuracy)

# Calculate and print the average accuracy across all folds
avg_accuracy = sum(accuracies) / len(accuracies)
print("Average accuracy:", avg_accuracy)


cuda
Average accuracy: 0.7187921727395412


In [27]:
import numpy as np

importance = ['must not to be', 'is better not be', 'doesn\'t matter', 'is better be', 'must be', 'must be']


def _tt_to_reason(TT : Tensor, importance : list):
    tt = torch.arctan(TT)
    tt = tt / (np.pi / 4)
    tt = tt + 1
    tt = tt / 2
    tt = tt * (len(importance) - 1)
    tt = tt.to(torch.int)
    reason = []
    rules = model.tt.shape[2]
    features = model.tt.shape[1]
    for i in range(rules):
        reason.append(list())
        for j in range(features):
            degree = importance[tt[0][j][i] % len(importance)]
            reason[i].append(degree)
    return reason
    
tt = torch.clamp(model.tt, -1, +1)
print(tt)
print(_tt_to_reason(tt, importance))

def explain(model : FQ):
    rules = model.mean.shape[2]
    features = model.mean.shape[1]
    degrees = _tt_to_reason(tt, importance)
    for i in range(rules):
        print(f'Rule {i}:')
        flag = False
        for j in range(features):
            if degrees[i][j] == 'doesn\'t matter':
                continue
            max_ = model.mean[0][j][i] + model.std[0][j][i]
            min_ = model.mean[0][j][i] - model.std[0][j][i]
            if flag:
                print('\tand')
            print(f'\tfeature number {j} {degrees[i][j]} between {min_:.3f} and {max_:.3f}') 
            flag = True
            
explain(model)  
              

tensor([[[-1.,  1., -1.,  1.],
         [-1., -1., -1., -1.],
         [ 1.,  1., -1., -1.],
         [-1., -1.,  1., -1.]]], device='cuda:0', grad_fn=<ClampBackward1>)
[['must not to be', 'must not to be', 'must be', 'must not to be'], ['must be', 'must not to be', 'must be', 'must not to be'], ['must not to be', 'must not to be', 'must not to be', 'must be'], ['must be', 'must not to be', 'must not to be', 'must not to be']]
Rule 0:
	feature number 0 must not to be between 1.262 and 1.294
	and
	feature number 1 must not to be between -0.083 and 0.140
	and
	feature number 2 must be between -3.588 and 4.887
	and
	feature number 3 must not to be between 0.320 and 0.703
Rule 1:
	feature number 0 must be between -3.416 and 3.782
	and
	feature number 1 must not to be between 0.181 and 0.236
	and
	feature number 2 must be between -0.148 and 0.303
	and
	feature number 3 must not to be between 0.288 and 0.475
Rule 2:
	feature number 0 must not to be between 1.234 and 1.218
	and
	feature numbe

In [24]:
print(model.linear.weight)

Parameter containing:
tensor([[ 0.5806, -0.7438,  0.4040,  2.0087,  1.8524, -1.7726, -2.7667,  1.0223,
         -3.2103, -2.0972],
        [-2.9017,  1.7884,  1.9342, -0.1251, -1.5569, -0.7712,  1.9729, -3.8289,
          0.8894, -2.5925],
        [ 1.7760, -2.2826, -2.8452, -3.0591, -2.4998,  1.9782, -1.1152,  1.8032,
          0.6865,  2.7704]], device='cuda:0', requires_grad=True)
